In [27]:
from pathlib import Path
import re

import numpy as np
import pyvista as pv


def read_off(filename):
    with open(filename, "r") as f:
        lines = []

        for line in f:
            line = line.split("#", 1)[0].strip()
            if line:
                lines.append(line)

    if lines[0] != "OFF":
        raise ValueError(f"{filename} is not a valid OFF file")

    n_vertices, n_faces, _ = map(int, lines[1].split()[:3])

    vertices = np.array(
        [list(map(float, lines[i].split()[:3])) for i in range(2, 2 + n_vertices)]
    )

    faces = []
    index = 2 + n_vertices

    for _ in range(n_faces):
        values = lines[index].split()
        n = int(values[0])

        face = [int(x) for x in values[1 : n + 1]]
        faces.extend([n] + face)

        index += 1

    return pv.PolyData(vertices, np.array(faces, dtype=np.int64))

In [29]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm


folder = Path("./../OFF_files")

plotter = pv.Plotter()

files = sorted(folder.glob("try*.off"))

# Extract component numbers
components = []
for f in files:
    match = re.fullmatch(r"try(\d+)\.OFF", f.name, re.IGNORECASE)

    if match:
        components.append(int(match.group(1)))


# --------------------------------------------------
# Discrete colors
# --------------------------------------------------

n_colors = len(components)

cmap = plt.get_cmap("tab20", n_colors)

# Boundaries between discrete values
boundaries = np.arange(n_colors + 1) - 0.5
norm = BoundaryNorm(boundaries, n_colors)

# --------------------------------------------------
# Plot
# --------------------------------------------------

plotter = pv.Plotter()

for off_file in files:
    match = re.fullmatch(
        r"try(\d+)\.OFF",
        off_file.name,
        re.IGNORECASE,
    )

    if match is None:
        continue

    number = int(match.group(1))

    # Index into the discrete color list
    color_index = components.index(number)

    mesh = read_off(off_file)

    # Same value for every face in this component
    mesh.cell_data["component"] = np.full(
        mesh.n_cells,
        color_index,
    )

    plotter.add_mesh(
        mesh,
        scalars="component",
        cmap=cmap,
        clim=[-0.5, n_colors - 0.5],
        show_edges=False,
    )


plotter.show()

Widget(value='<iframe src="http://localhost:62056/index.html?ui=P_0x32795e850_23&reconnect=auto" class="pyvist…